# Inventory matching (assign `games_db.id` to `inventory`)

This notebook matches game titles from `inventory.csv` to `games_db.csv`.

False-positive control (high level):
- Filter by `console_name` first (exact match).
- Use `rapidfuzz` as a conservative matcher.
- Only call an **LLM** (OpenAI or **xAI Grok**) as a reranker for rare/ambiguous cases.

LLM rerank rule (same for both providers):
- Call the LLM if fuzzy rejected and either:
  - `top1_score` is in **[50, 85)**, or
  - `top1` is ambiguous (maps to multiple ids) and `top1_score >= 85`.

**Provider:** set env `MATCH_LLM_PROVIDER` to `openai` (default) or `grok` (alias: `xai`). See the env/secrets cell after `%pip install`.

Outputs (written next to this notebook):
- `inventory_matches.csv`
- `inventory_pending.csv` (includes `match_reason`)


In [ ]:
%pip install pandas rapidfuzz openai

### LLM provider configuration

| Variable | Default | Meaning |
|----------|---------|---------|
| `MATCH_LLM_PROVIDER` | `openai` | `openai` or `grok` / `xai` |
| **OpenAI** | | |
| `OPENAI_API_KEY` | — | API key (or use Databricks secrets) |
| `OPENAI_SECRET_SCOPE` | `openai-api` | Databricks secret scope |
| `OPENAI_SECRET_KEY_NAME` | `api_key` | Secret key name |
| `OPENAI_MODEL` | `gpt-4o-mini` | Chat model |
| **Grok (xAI)** | | Same Python `openai` package, `base_url=https://api.x.ai/v1` |
| `XAI_API_KEY` | — | API key (or secrets below) |
| `GROK_SECRET_SCOPE` | `grok-api` | Databricks scope (see `advanced_forecast_weekly`) |
| `GROK_SECRET_KEY` | `grok_api_key` | Secret key name |
| `GROK_MODEL` / `XAI_MODEL` | `grok-4-1-fast-reasoning` | Chat model |

List secret scopes in a notebook: `dbutils.secrets.listScopes()`.


In [ ]:
from __future__ import annotations

import json
import os
import re
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI
from rapidfuzz import fuzz, process

# --- Paths (inputs and outputs live next to this notebook) ---
GAMES_DB_CSV = Path("games_db.csv")
INVENTORY_CSV = Path("inventory.csv")

OUT_MATCHES = Path("inventory_matches.csv")
OUT_PENDING = Path("inventory_pending.csv")

# --- Matching thresholds ---
THRESHOLD_HIGH = 92
REJECT_GAP = 7  # accept only if top-1 beats top-2 by at least this amount

SLEEP_SECONDS = 0.25  # sleep before each LLM request to reduce rate limits (RPM-style)

# --- LLM provider: openai | grok (alias xai) ---
_raw_provider = os.environ.get("MATCH_LLM_PROVIDER", "openai").strip().lower()
MATCH_LLM_PROVIDER = "grok" if _raw_provider in ("grok", "xai") else "openai"

# --- Rerank settings (used for both providers) ---
OPENAI_MODEL = os.environ.get("OPENAI_MODEL", "gpt-4o-mini")
GROK_MODEL = os.environ.get("GROK_MODEL", os.environ.get("XAI_MODEL", "grok-4-1-fast-reasoning"))
OPENAI_TOPK_NORMS = int(os.environ.get("OPENAI_TOPK_NORMS", "20"))
OPENAI_MAX_CANDIDATES = int(os.environ.get("OPENAI_MAX_CANDIDATES", "60"))
OPENAI_MIN_SCORE = 50
OPENAI_MAX_SCORE_EXCL = 85

# --- Input columns ---
EXPECTED_GAMES_COLS = ["id", "console_name_source", "product-name_source"]
EXPECTED_INV_COLS = ["console_name_target", "title_name_target"]


def get_openai_api_key() -> str:
    k = os.environ.get("OPENAI_API_KEY", "").strip()
    if k:
        return k

    dbutils_obj = globals().get("dbutils")
    if dbutils_obj is None:
        raise RuntimeError(
            "OPENAI_API_KEY not set and dbutils.secrets is not available. "
            "Attach the notebook to a Databricks cluster or configure secrets."
        )

    scope = os.environ.get("OPENAI_SECRET_SCOPE", "openai-api")
    key_name = os.environ.get("OPENAI_SECRET_KEY_NAME", "api_key")
    try:
        return dbutils_obj.secrets.get(scope=scope, key=key_name)
    except Exception as e:
        raise RuntimeError(
            f"Failed to read secret scope={scope!r} key={key_name!r}: {e!r}"
        ) from e


def get_grok_api_key() -> str:
    k = os.environ.get("XAI_API_KEY", "").strip()
    if k:
        return k

    dbutils_obj = globals().get("dbutils")
    if dbutils_obj is None:
        raise RuntimeError(
            "XAI_API_KEY not set and dbutils.secrets is not available. "
            "Attach the notebook to a Databricks cluster or configure secrets."
        )

    scope = os.environ.get("GROK_SECRET_SCOPE", "grok-api")
    key_name = os.environ.get("GROK_SECRET_KEY", "grok_api_key")
    try:
        return dbutils_obj.secrets.get(scope=scope, key=key_name)
    except Exception as e:
        raise RuntimeError(
            f"Failed to read Grok secret scope={scope!r} key={key_name!r}: {e!r}"
        ) from e


def get_llm_client_and_model() -> tuple[OpenAI, str, str]:
    """Returns (client, model_name, reason_tag) where reason_tag is openai or grok."""
    if MATCH_LLM_PROVIDER == "grok":
        return (
            OpenAI(api_key=get_grok_api_key(), base_url="https://api.x.ai/v1"),
            GROK_MODEL,
            "grok",
        )
    return OpenAI(api_key=get_openai_api_key()), OPENAI_MODEL, "openai"


llm_client, LLM_MODEL, LLM_REASON_TAG = get_llm_client_and_model()

print(f"LLM provider={MATCH_LLM_PROVIDER!r}  model={LLM_MODEL!r}")


if not GAMES_DB_CSV.exists():
    raise FileNotFoundError(f"Missing {GAMES_DB_CSV}. Put games_db.csv next to this notebook.")
if not INVENTORY_CSV.exists():
    raise FileNotFoundError(f"Missing {INVENTORY_CSV}. Put inventory.csv next to this notebook.")

games_df = pd.read_csv(GAMES_DB_CSV, encoding="utf-8-sig")
inventory_df = pd.read_csv(INVENTORY_CSV, encoding="utf-8-sig")

games_df.columns = games_df.columns.str.strip()
inventory_df.columns = inventory_df.columns.str.strip()

missing_games = [c for c in EXPECTED_GAMES_COLS if c not in games_df.columns]
missing_inv = [c for c in EXPECTED_INV_COLS if c not in inventory_df.columns]
if missing_games:
    raise ValueError(f"games_db.csv missing columns: {missing_games}. Found: {list(games_df.columns)}")
if missing_inv:
    raise ValueError(f"inventory.csv missing columns: {missing_inv}. Found: {list(inventory_df.columns)}")

# Normalize console columns (exact match filter)
games_df["console_name_source"] = games_df["console_name_source"].astype(str).str.strip()
inventory_df["console_name_target"] = inventory_df["console_name_target"].astype(str).str.strip()

# Make id consistently string
games_df["id"] = games_df["id"].astype(str).str.strip()
games_df = games_df[games_df["id"] != ""]

print(f"Loaded games_db rows={len(games_df)}  inventory rows={len(inventory_df)}")


In [ ]:
def normalize_alpha_num(s: object) -> str:
    """Lowercase + keep only alphanumeric; collapse into space-separated tokens."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return ""
    s = str(s).lower()
    s = re.sub(r"[^a-z0-9]+", " ", s)
    return " ".join(s.split())


# Pre-index games_db by console for speed and for strict id mapping.
games_df["product_norm"] = games_df["product-name_source"].apply(normalize_alpha_num)

id_to_title = games_df.groupby("id")["product-name_source"].first().astype(str).to_dict()

console_index: dict[str, dict[str, object]] = {}
for console_name, grp in games_df.groupby("console_name_source"):
    grp = grp[grp["product_norm"].astype(str).str.len() > 0]
    if grp.empty:
        continue

    norm_to_ids = (
        grp.groupby("product_norm")["id"]
        .apply(lambda s: sorted(set(map(str, s.tolist()))))
        .to_dict()
    )
    choices_norm = list(norm_to_ids.keys())

    console_index[console_name] = {
        "choices_norm": choices_norm,
        "norm_to_ids": norm_to_ids,
    }


def fuzzy_top2(console_name: str, query_norm: str):
    entry = console_index.get(console_name)
    if entry is None:
        return None

    choices_norm = entry["choices_norm"]  # type: ignore[assignment]
    norm_to_ids = entry["norm_to_ids"]  # type: ignore[assignment]

    if not choices_norm:
        return None

    res = process.extract(query_norm, choices_norm, scorer=fuzz.token_set_ratio, limit=2)
    if not res:
        return None

    top1_choice = res[0][0]
    top1_score = int(res[0][1])
    if len(res) > 1:
        top2_score = int(res[1][1])
    else:
        top2_score = 0

    ids_for_top1 = norm_to_ids.get(top1_choice, [])
    ambiguous = len(ids_for_top1) != 1

    return {
        "top1_choice": top1_choice,
        "top1_score": top1_score,
        "top2_score": top2_score,
        "ids_for_top1": ids_for_top1,
        "ambiguous": ambiguous,
    }


def fuzzy_accept(fuzzy_info: dict) -> bool:
    if fuzzy_info["top1_score"] < THRESHOLD_HIGH:
        return False
    if (fuzzy_info["top1_score"] - fuzzy_info["top2_score"]) < REJECT_GAP:
        return False
    return len(fuzzy_info["ids_for_top1"]) == 1


def llm_rerank(
    client: OpenAI,
    model: str,
    reason_tag: str,
    console_name: str,
    inv_title: str,
    candidates: list[dict],
):
    """Returns (matched_id or None, confidence or None, pending_reason_code)."""
    invalid = f"{reason_tag}_invalid_response"
    no_match = f"{reason_tag}_no_match"
    low_conf = f"{reason_tag}_low_confidence"

    candidate_ids = {str(c["candidate_id"]).strip() for c in candidates}

    system_prompt = (
        "You are a strict matching assistant for used video game listings. "
        "Given a console and an inventory title, choose the candidate_id that refers to the same game on that console. "
        "If you are not confident the titles refer to the same game (or if any candidate looks like a different game), "
        "return matched_id as null and confidence as low. "
        "Do not guess."
    )

    user_prompt = {
        "console_name_target": console_name,
        "title_name_target": inv_title,
        "candidates": candidates,
        "instructions": [
            "Return JSON only.",
            "Select at most one candidate.",
            "If no candidate is a clear match, use matched_id=null and confidence=low."
        ],
    }

    time.sleep(SLEEP_SECONDS)

    completion = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": json.dumps(user_prompt, ensure_ascii=False)},
        ],
        response_format={"type": "json_object"},
        temperature=0,
    )

    text = completion.choices[0].message.content
    if not text:
        return None, None, invalid

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        return None, None, invalid

    matched_id = data.get("matched_id", None)
    confidence = data.get("confidence", None)

    if matched_id is None or str(matched_id).strip().lower() == "null":
        return None, str(confidence or "low"), no_match

    matched_id_str = str(matched_id).strip()
    if matched_id_str not in candidate_ids:
        return None, str(confidence or "low"), no_match

    confidence_str = str(confidence or "low").strip().lower()
    if confidence_str != "high":
        return matched_id_str, confidence_str, low_conf

    return matched_id_str, confidence_str, ""


In [ ]:
matches_rows: list[dict] = []
pending_rows: list[dict] = []

openai_calls = 0
openai_accepts = 0
grok_calls = 0
grok_accepts = 0

# Cache by (console_name_target, normalized_title)
cache: dict[tuple[str, str], dict] = {}

total = len(inventory_df)
for i, row in inventory_df.iterrows():
    console_name = str(row["console_name_target"]).strip()
    inv_title = row["title_name_target"]
    base = row.to_dict()

    query_norm = normalize_alpha_num(inv_title)
    cache_key = (console_name, query_norm)

    if cache_key in cache:
        outcome = cache[cache_key]
    else:
        if not query_norm:
            outcome = {
                "matched_id": None,
                "match_score": None,
                "match_method": None,
                "match_reason": "empty_title",
            }
        else:
            fuzzy_info = fuzzy_top2(console_name, query_norm)
            if fuzzy_info is None:
                outcome = {
                    "matched_id": None,
                    "match_score": None,
                    "match_method": None,
                    "match_reason": "no_console_candidates",
                }
            else:
                if fuzzy_accept(fuzzy_info):
                    outcome = {
                        "matched_id": fuzzy_info["ids_for_top1"][0],
                        "match_score": fuzzy_info["top1_score"],
                        "match_method": "fuzzy",
                        "match_reason": "",
                    }
                else:
                    top1_score = fuzzy_info["top1_score"]
                    ambiguous = fuzzy_info["ambiguous"]

                    should_llm = (
                        (top1_score >= OPENAI_MIN_SCORE and top1_score < OPENAI_MAX_SCORE_EXCL)
                        or (ambiguous and top1_score >= OPENAI_MAX_SCORE_EXCL)
                    )

                    if should_llm:
                        if LLM_REASON_TAG == "grok":
                            grok_calls += 1
                        else:
                            openai_calls += 1

                        entry = console_index.get(console_name)
                        choices_norm = entry["choices_norm"]  # type: ignore[assignment]
                        norm_to_ids = entry["norm_to_ids"]  # type: ignore[assignment]

                        resK = process.extract(
                            query_norm,
                            choices_norm,
                            scorer=fuzz.token_set_ratio,
                            limit=OPENAI_TOPK_NORMS,
                        )

                        candidate_map: dict[str, str] = {}
                        if resK:
                            for cand_norm, _, _ in resK:
                                ids = norm_to_ids.get(cand_norm, [])
                                for cid in ids:
                                    cid_str = str(cid).strip()
                                    if cid_str and cid_str not in candidate_map:
                                        candidate_map[cid_str] = id_to_title.get(cid_str, "")
                                if len(candidate_map) >= OPENAI_MAX_CANDIDATES:
                                    break

                        candidates = [
                            {"candidate_id": cid, "candidate_title": title}
                            for cid, title in candidate_map.items()
                        ]

                        if not candidates:
                            outcome = {
                                "matched_id": None,
                                "match_score": None,
                                "match_method": None,
                                "match_reason": "no_candidates",
                            }
                        else:
                            matched_id, confidence, pending_reason = llm_rerank(
                                llm_client,
                                LLM_MODEL,
                                LLM_REASON_TAG,
                                console_name=console_name,
                                inv_title=str(inv_title),
                                candidates=candidates,
                            )

                            rerank_method = (
                                "grok_rerank" if LLM_REASON_TAG == "grok" else "openai_rerank"
                            )
                            default_no_match = (
                                "grok_no_match" if LLM_REASON_TAG == "grok" else "openai_no_match"
                            )

                            if matched_id is not None and pending_reason == "":
                                if LLM_REASON_TAG == "grok":
                                    grok_accepts += 1
                                else:
                                    openai_accepts += 1
                                outcome = {
                                    "matched_id": matched_id,
                                    "match_score": top1_score,
                                    "match_method": rerank_method,
                                    "match_reason": "",
                                }
                            else:
                                outcome = {
                                    "matched_id": None,
                                    "match_score": None,
                                    "match_method": None,
                                    "match_reason": pending_reason or default_no_match,
                                }
                    else:
                        if top1_score < OPENAI_MIN_SCORE:
                            if ambiguous:
                                outcome = {
                                    "matched_id": None,
                                    "match_score": None,
                                    "match_method": None,
                                    "match_reason": "ambiguous_top1_rejected",
                                }
                            else:
                                outcome = {
                                    "matched_id": None,
                                    "match_score": None,
                                    "match_method": None,
                                    "match_reason": "below_openai_band",
                                }
                        else:
                            outcome = {
                                "matched_id": None,
                                "match_score": None,
                                "match_method": None,
                                "match_reason": "gap_too_small",
                            }

        cache[cache_key] = outcome

    matched_id = outcome.get("matched_id")
    match_score = outcome.get("match_score")
    match_method = outcome.get("match_method")

    if matched_id is None or match_score is None or not match_method:
        out = dict(base)
        out["match_reason"] = outcome.get("match_reason") or "no_match"
        pending_rows.append(out)
    else:
        out = dict(base)
        out["matched_id"] = str(matched_id).strip()
        out["match_score"] = int(match_score)
        out["match_method"] = match_method
        matches_rows.append(out)

    if (i + 1) % 2000 == 0:
        print(
            f"Progress {i+1}/{total}  fuzzy_matches={len(matches_rows)}  pending={len(pending_rows)}  "
            f"openai_calls={openai_calls} grok_calls={grok_calls}"
        )

matches_df = pd.DataFrame(matches_rows)
pending_df = pd.DataFrame(pending_rows)
print(
    f"Done. matches={len(matches_df)} pending={len(pending_df)}  "
    f"openai_calls={openai_calls} openai_accepts={openai_accepts}  "
    f"grok_calls={grok_calls} grok_accepts={grok_accepts}"
)


In [ ]:
# Write outputs
matches_df.to_csv(OUT_MATCHES, index=False, encoding="utf-8-sig")
pending_df.to_csv(OUT_PENDING, index=False, encoding="utf-8-sig")

print(f"Wrote: {OUT_MATCHES}")
print(f"Wrote: {OUT_PENDING}")


In [ ]:
# Report + basic consistency check
total = len(inventory_df)
accepted = len(matches_df)
pending = len(pending_df)
pct = (accepted / total * 100.0) if total else 0.0

games_ids = set(games_df["id"].astype(str).tolist())
bad_ids = 0
if not matches_df.empty and "matched_id" in matches_df.columns:
    bad_ids = sum(1 for x in matches_df["matched_id"].astype(str).tolist() if x not in games_ids)

reason_counts = {}
if not pending_df.empty and "match_reason" in pending_df.columns:
    reason_counts = pending_df["match_reason"].value_counts().to_dict()

report = {
    "total_inventory": int(total),
    "accepted_matches": int(accepted),
    "pending": int(pending),
    "pct_complete": round(pct, 2),
    "threshold_high": THRESHOLD_HIGH,
    "reject_gap": REJECT_GAP,
    "match_llm_provider": MATCH_LLM_PROVIDER,
    "openai_calls": int(openai_calls),
    "openai_accepts": int(openai_accepts),
    "grok_calls": int(grok_calls),
    "grok_accepts": int(grok_accepts),
    "matched_ids_not_found_in_games_db": int(bad_ids),
    "pending_match_reason_counts": reason_counts,
}

print(report)

if not pending_df.empty and "match_reason" in pending_df.columns:
    print("Top pending reasons:")
    print(pending_df["match_reason"].value_counts().head(20))
